# Line Emission Denoising — Sweep-Winner Validation + Artifact Diagnostics

Per the 2026-06-18 mentor pivot: full-image denoising of line-emission velocity channels
(NO patches), channel-by-channel, cube-level holdout, `bettermoments` M0/M1/M2 as the actual
scientific deliverable.

**What this run answers.** The 12-run random sweep (results committed:
`results/sweep_results.csv`) found a config at **PSNR 37.11 dB** — +4.16 dB over the V12
reference. But the beam A/B in the same session showed a pixel-metric win (+1.24 dB) that came
with **worse** M0 moment-map reliability (+59.8%±33.0% vs V12's +69.8%±15.2%). So a PSNR number
alone cannot be trusted as an improvement. This notebook:

1. **Retrains the sweep winner** (base=48, mult 1×2×4×8, lr=8.2e-4, alpha=0.888, no beam,
   batch 16) and runs it through the **same all-5-holdout moment-map protocol** V12 was judged
   on. That comparison decides whether the sweep winner becomes the new reference checkpoint.
2. **Quantifies the two known artifacts across every validation channel** — peak overshoot and
   low-SNR invented structure ("hallucination") — replacing the single channel-100 anecdote
   with statistics.

**Reference to beat — V12** (`unet_line_emission_continuum_best.pth`, fixed 30 epochs):
PSNR 32.95 dB | SSIM 0.9857 | MSE 0.000681; 5-cube holdout **M0 +69.8%±15.2% | M1 +17.5%±7.8%
| M2 +20.1%±14.3%** (all 5 cubes positive on all 3 moments).

Everything else is held identical to V12 and to the sweep: 14 cubes / 11 RunIDs, cube-level
split (3 RunID groups = 5 cubes held out, inference only), Gaussian channel sampling (center
100), 256×256 full images, continuum subtraction (n=5), shared **dirty**-scale per-channel
normalisation (invertible at inference), linear output head, seed=42.

**Kaggle setup:** GPU on, Internet on, `Add Input` → line-emission Dataset (FITS cubes).
The bootstrap locates it under `/kaggle/input/`.

**DDPM lives in `06-ddpm-line-emission.ipynb`** — separate notebook, never touches these files.

## 0. Bootstrap (clone repo for `src/`, locate data)

In [ ]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
if ON_KAGGLE:
    REPO='/kaggle/working/EXXA'; PKG=os.path.join(REPO,'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch','line-emission','--depth','1','https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin','line-emission'], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/line-emission'], check=True)
    # pytorch-msssim for the SSIM loss; bettermoments for moment-map evaluation (Sec. 6)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks'));  sys.path.insert(0, PKG)
    # find the line-emission data dir under /kaggle/input (contains run_* subfolders)
    hits = glob.glob('/kaggle/input/**/*_dirty.fits', recursive=True)
    DATA_DIR = os.path.dirname(os.path.dirname(hits[0])) if hits else None
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'): os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../data/Line Emission Data'
print('cwd:', os.getcwd(), '| DATA_DIR:', DATA_DIR)

## 0b. Pull latest code (re-run anytime — NO kernel restart needed)

After pushing new changes to the `line-emission` branch, re-run this cell to fetch them and
hot-reload the `src/` modules, then re-run the imports cell below.

In [ ]:
# Pull latest from the line-emission branch and hot-reload src/ (no kernel restart).
import os, sys, subprocess

ON_KAGGLE = os.path.exists('/kaggle')
REPO = '/kaggle/working/EXXA'

if ON_KAGGLE and os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', 'line-emission'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/line-emission'], check=True)
    print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                         capture_output=True, text=True).stdout.strip())
elif ON_KAGGLE:
    print('repo not cloned yet -- run the bootstrap cell (0.) first')

for _m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]
print('src.* cleared from module cache -- now re-run the imports cell below.')

## 1. Imports, device, config

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from src.data.cube_split import split_cubes
from src.data.fits_cube_dataset import FITSChannelDataset, continuum_of
from src.models.unet import UNet
from src.training.sweep import train_unet, val_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPU  = torch.cuda.device_count()
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('device:', device, '| GPUs:', N_GPU,
      '->', [torch.cuda.get_device_name(i) for i in range(N_GPU)] if N_GPU else 'cpu')

TARGET_SIZE = 256
N_SAMPLES   = 50      # identical to V12 and to the sweep -> numbers stay comparable
NW          = 4 if ON_KAGGLE else 0   # FITS I/O bottleneck; workers help on Kaggle

# V12 reference (fixed 30-epoch run) -- the bar this notebook is measured against
V12 = {'psnr': 32.95, 'ssim': 0.9857, 'mse': 0.000681,
       'M0': (69.8, 15.2), 'M1': (17.5, 7.8), 'M2': (20.1, 14.3)}
# Sweep winner as scored during the sweep (results/sweep_results.csv, run 7)
SWEEP_WINNER_PSNR = 37.109

## 2. Cube-level split (3 RunID groups held out for inference only)

In [ ]:
train_cubes, val_cubes, holdout_cubes = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                     val_fraction=0.2, seed=SEED)

## 3. Datasets (full-image 256×256, continuum-subtracted, shared dirty-scale norm)

`return_beam=False` — the sweep winner does not use beam conditioning (beam correlated
*negatively* with PSNR across the 12 sweep runs, r=-0.33, and all top-5 configs had
`use_beam=False`).

In [ ]:
SUBTRACT_CONTINUUM = True
CONTINUUM_N        = 5
train_ds = FITSChannelDataset(train_cubes, n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
                              subtract_continuum=SUBTRACT_CONTINUUM, continuum_n=CONTINUUM_N)
val_ds   = FITSChannelDataset(val_cubes,   n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
                              subtract_continuum=SUBTRACT_CONTINUUM, continuum_n=CONTINUUM_N)
print('train items:', len(train_ds), '| val items:', len(val_ds))

## 4. Retrain the sweep winner

Exact config from `results/sweep_results.csv` run 7 (best of 12 by PSNR). Early stopping is the
mentor-specified schedule (min 20 / max 60 / patience 5 on val loss, best-epoch weights
restored). Expect ≈37.1 dB — a large deviation would mean the sweep result was seed luck rather
than a real config effect, which is itself worth knowing before trusting it.

In [ ]:
WINNER = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
              lr=0.0008196504330730313, alpha=0.8877681051398497,
              sched_patience=8, use_beam=False, batch_size=16)
CKPT_WINNER = '../results/checkpoints/unet_line_emission_sweepwinner_best.pth'
print('winner config:', WINNER)

t0 = time.time()
res = train_unet(train_ds, val_ds, device, **WINNER,
                 min_epochs=20, max_epochs=60, patience=5,
                 num_workers=NW, seed=SEED, ckpt_path=CKPT_WINNER, verbose=True)
print(f'\ntrained in {time.time()-t0:.0f}s')
print('winner: PSNR {:.4f} | SSIM {:.4f} | MSE {:.6f} | best ep {} ({} run)'.format(
    res['psnr'], res['ssim'], res['mse'], res['best_epoch'], res['epochs_run']))
print('sweep-time PSNR was {:.3f} dB -> reproduction delta {:+.3f} dB'.format(
    SWEEP_WINNER_PSNR, res['psnr'] - SWEEP_WINNER_PSNR))
print('vs V12     : PSNR {:+.3f} dB | SSIM {:+.5f} | MSE {:+.6f}'.format(
    res['psnr'] - V12['psnr'], res['ssim'] - V12['ssim'], res['mse'] - V12['mse']))

## 5. Loss curve

In [ ]:
tr_hist, va_hist = res['train_losses'], res['val_losses']
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(tr_hist)+1), tr_hist, marker='o', ms=3, label='train')
plt.plot(range(1, len(va_hist)+1), va_hist, marker='s', ms=3, label='val')
be = int(np.argmin(va_hist)) + 1
plt.axvline(be, color='gray', ls=':')
plt.scatter([be], [min(va_hist)], color='#E8715A', zorder=5, label=f'best ep {be}')
plt.xlabel('epoch'); plt.ylabel('HybridLoss')
plt.title('Sweep winner (base 48, 1x2x4x8, alpha {:.3f})'.format(WINNER['alpha']))
plt.legend(); plt.grid(alpha=0.3); plt.yscale('log'); plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/sweepwinner_loss.png', dpi=140); plt.show()

## 6. All-5-Holdout moment maps — the decisive comparison

Identical contract to V12's evaluation: denoise **every** channel of all 5 held-out cubes at
full 600×600 (model runs at 256×256, output resampled back), rebuild the cube, write FITS, then
compare `bettermoments` M0/M1/M2 against clean — scoring improvement over the dirty baseline.
Per project rule, the mean over all 5 cubes with a std is the only trustworthy number; a single
cube is noise (the V7/V9 lesson).

Verdict logic: the winner replaces V12 as reference **only** if it holds M0 without inflating
variance. A PSNR win with M0 regression is the beam outcome, not an improvement.

In [ ]:
import csv
import matplotlib.ticker as mticker
import bettermoments as bm
from astropy.io import fits
from src.evaluation.moment_maps import generate_moment_maps

OUT_DIR = '../results'
BS = 32
os.makedirs(OUT_DIR, exist_ok=True)

ckpt = torch.load(CKPT_WINNER, map_location=device, weights_only=False)
eval_net = UNet(in_channels=1, out_channels=1, base_channels=ckpt['base_channels'],
                channel_multipliers=ckpt['channel_multipliers'], time_emb_dim=128,
                num_res_blocks=2, groups=math.gcd(8, ckpt['base_channels']),
                beam_dim=ckpt.get('beam_dim', 0)).to(device)
eval_net.load_state_dict(ckpt['model_state_dict']); eval_net.eval()
print('winner checkpoint loaded: epoch', ckpt.get('epoch'), '| val_loss',
      round(float(ckpt.get('val_loss', float('nan'))), 4))

def mdiff(a, b):
    mask = np.isfinite(a) & np.isfinite(b)
    return float(np.nanmean(np.abs(a[mask] - b[mask])))

def denoise_cube(ho_entry, net):
    """Full-cube inference: continuum-subtract, per-channel dirty-scale norm, denoise, invert."""
    with fits.open(ho_entry['dirty'], memmap=False) as hdul:
        dirty_raw = np.ascontiguousarray(hdul[0].data).astype(np.float32)
        hdr = hdul[0].header.copy()
    C, H, W = dirty_raw.shape
    dcont = continuum_of(dirty_raw, CONTINUUM_N)
    dirty_csub = dirty_raw - dcont[None, :, :]
    los  = dirty_csub.reshape(C, -1).min(axis=1)
    his  = dirty_csub.reshape(C, -1).max(axis=1)
    rngs = his - los
    norm = np.zeros_like(dirty_csub)
    nz = rngs > 0
    norm[nz] = (dirty_csub[nz] - los[nz, None, None]) / rngs[nz, None, None]

    denoised_csub = np.empty_like(dirty_csub)
    with torch.no_grad():
        for s in range(0, C, BS):
            t = torch.from_numpy(norm[s:s+BS])[:, None].float().to(device)
            t256 = F.interpolate(t, (TARGET_SIZE, TARGET_SIZE), mode='bilinear', align_corners=False)
            tz = torch.zeros(t256.size(0), dtype=torch.long, device=device)
            out = net(t256, tz)
            out600 = F.interpolate(out, (H, W), mode='bilinear', align_corners=False)[:, 0].cpu().numpy()
            for k in range(out600.shape[0]):
                ch = s + k
                denoised_csub[ch] = (out600[k] * rngs[ch] + los[ch]) if rngs[ch] > 0 else \
                                    np.full((H, W), los[ch], np.float32)
    out_path = os.path.join(OUT_DIR, 'denoised_sweepwinner_' + ho_entry['folder'] + '.fits')
    fits.writeto(out_path, denoised_csub.astype(np.float32), header=hdr, overwrite=True)
    return out_path, dirty_csub

rows_data = []
col_w = 30
hdr_line = '{:<{w}} {:>10} {:>10} {:>10} {:>10} {:>10} {:>10}'.format(
    'cube', 'dirty_M0', 'imp_M0%', 'dirty_M1', 'imp_M1%', 'dirty_M2', 'imp_M2%', w=col_w)
print('\n' + hdr_line); print('-' * len(hdr_line))

for ho in holdout_cubes:
    print('  denoising', ho['folder'], '...', end=' ', flush=True)
    out_fits_ho, dirty_csub = denoise_cube(ho, eval_net)
    print('done')
    with fits.open(ho['clean'], memmap=False) as h:
        clean_raw = np.ascontiguousarray(h[0].data).astype(np.float32)
    ccont = continuum_of(clean_raw, CONTINUUM_N)
    clean_csub = clean_raw - ccont[None]
    _, velax = bm.load_cube(ho['dirty'])
    c0, c1, c2 = generate_moment_maps(None, data_velax=(clean_csub, velax))
    d0, d1, d2 = generate_moment_maps(None, data_velax=(dirty_csub, velax))
    n0, n1, n2 = generate_moment_maps(out_fits_ho)
    row = {'cube': ho['folder']}
    for nm, cl, di, no in [('M0', c0, d0, n0), ('M1', c1, d1, n1), ('M2', c2, d2, n2)]:
        dd = mdiff(cl, di); nn = mdiff(cl, no)
        imp = 100.0 * (1 - nn / dd) if dd > 0 else float('nan')
        row['dirty_' + nm] = round(dd, 6); row['imp_' + nm] = round(imp, 2)
    rows_data.append(row)
    print('{:<{w}} {:>10.4g} {:>9.1f}% {:>10.4g} {:>9.1f}% {:>10.4g} {:>9.1f}%'.format(
        ho['folder'], row['dirty_M0'], row['imp_M0'], row['dirty_M1'], row['imp_M1'],
        row['dirty_M2'], row['imp_M2'], w=col_w))

moments = ['M0', 'M1', 'M2']
imps  = {m: [r['imp_'+m] for r in rows_data if r['imp_'+m] == r['imp_'+m]] for m in moments}
means = {m: float(np.mean(imps[m])) for m in moments}
stds  = {m: float(np.std(imps[m], ddof=1)) if len(imps[m]) > 1 else 0.0 for m in moments}

print('\n' + '=' * len(hdr_line))
print('SWEEP-WINNER SUMMARY (n={} cubes)   vs   V12 baseline'.format(len(rows_data)))
for m in moments:
    v_mu, v_sd = V12[m]
    print('  {}: {:+.1f}% +/-{:.1f}%   |  V12 {:+.1f}% +/-{:.1f}%   |  delta {:+.1f} pts'.format(
        m, means[m], stds[m], v_mu, v_sd, means[m] - v_mu))
n_pos = sum(1 for r in rows_data if all(r['imp_'+m] > 0 for m in moments))
print('  cubes positive on all 3 moments: {}/{}  (V12: 5/5)'.format(n_pos, len(rows_data)))

# explicit verdict -- pixel-metric gain alone is NOT sufficient (beam A/B lesson)
m0_holds = means['M0'] >= V12['M0'][0] - 2.0
var_ok   = stds['M0'] <= V12['M0'][1] * 1.5
print('\nVERDICT: ' + (
    'sweep winner supersedes V12 as reference checkpoint (M0 held, variance not inflated)'
    if (m0_holds and var_ok and n_pos == len(rows_data)) else
    'NOT a clean improvement -- M0 mean and/or variance regressed; V12 stays the reference. '
    'Pixel metrics improved but the scientific deliverable did not (same pattern as the beam A/B).'))

csv_path = os.path.join(OUT_DIR, 'moment_map_holdout_summary_sweepwinner.csv')
fieldnames = ['cube', 'dirty_M0', 'imp_M0', 'dirty_M1', 'imp_M1', 'dirty_M2', 'imp_M2']
with open(csv_path, 'w', newline='') as cf:
    w = csv.DictWriter(cf, fieldnames=fieldnames); w.writeheader()
    for r in rows_data: w.writerow(r)
    w.writerow({'cube': 'MEAN', **{'imp_'+m: round(means[m], 2) for m in moments},
                **{'dirty_'+m: '' for m in moments}})
    w.writerow({'cube': 'STD',  **{'imp_'+m: round(stds[m], 2) for m in moments},
                **{'dirty_'+m: '' for m in moments}})
print('\nCSV saved ->', csv_path)

# grouped bar: winner vs V12 on all three moments
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(moments)); w_ = 0.38
ax.bar(x - w_/2, [means[m] for m in moments], w_, yerr=[stds[m] for m in moments], capsize=5,
       label='sweep winner', color='#4E91C7', alpha=0.9)
ax.bar(x + w_/2, [V12[m][0] for m in moments], w_, yerr=[V12[m][1] for m in moments], capsize=5,
       label='V12 reference', color='#9B9B9B', alpha=0.9)
for i, m in enumerate(moments):
    ax.scatter([i - w_/2]*len(imps[m]), imps[m], color='k', s=20, zorder=5, alpha=0.7)
ax.axhline(0, color='#333333', lw=0.8, ls='--')
ax.set_xticks(x); ax.set_xticklabels(['Moment 0\n(intensity)', 'Moment 1\n(velocity)',
                                      'Moment 2\n(dispersion)'])
ax.set_ylabel('Improvement over dirty (%)')
ax.set_title('Sweep winner vs V12 — moment-map improvement, 5 held-out cubes')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%+.0f%%'))
ax.legend(); ax.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'moment_map_sweepwinner_vs_v12.png'), dpi=140); plt.show()

## 7. Artifact diagnostics — peak overshoot + low-SNR invented structure

Both known artifacts have so far been documented from a **single** channel (channel 100), which
the V7/V9 variance lesson says is not enough. This measures them across **every** validation
channel:

- **Peak overshoot:** `denoised.max() / clean.max()` per channel. Prior single-channel figure
  was 1.151 (≈15% too bright). Reported here as a distribution.
- **Invented structure ("hallucination"):** for each channel, look only at pixels where clean is
  background (`clean < 10% of clean.max()`) and count how many the model pushed above 20% of
  clean's peak — signal asserted where the ground truth has none. Reported as an area fraction
  plus a connected-component count (how many distinct fake blobs), split by channel SNR so we
  can say *whether it is a low-SNR-specific failure*, which is the open question flagged to the
  mentor.

This is characterisation, not a fix — but it turns "hallucination observed once" into a number
the midterm report can state and later runs can be compared against.

In [ ]:
from src.evaluation.artifacts import channel_artifacts, summarise, INVENT_FRAC, BLOB_MIN_PX

eval_net.eval()
rows = []
with torch.no_grad():
    for ix in range(len(val_ds)):
        d, c = val_ds[ix]
        pred = eval_net(d[None].to(device),
                        torch.zeros(1, dtype=torch.long, device=device))[0, 0].cpu().numpy()
        cl, dt = c[0].numpy(), d[0].numpy()
        if float(cl.max()) <= 0:
            continue                      # empty channel: nothing to score against
        ci, ch = val_ds.index[ix]
        rows.append({'cube': val_ds.cube_paths[ci][2], 'channel': int(ch),
                     **channel_artifacts(cl, dt, pred)})

s = summarise(rows)
print('validation channels analysed:', s['n_channels'])
print('\nPEAK OVERSHOOT  denoised.max / clean.max   (1.0 = perfect; prior ch-100 figure 1.151)')
print('  mean {overshoot_mean:.3f} | median {overshoot_median:.3f} | p90 {overshoot_p90:.3f} | max {overshoot_max:.3f}'.format(**s))
print('  channels overshooting by >10%: {:.0f}%'.format(100 * s['frac_over_10pct']))
print('\nNEGATIVE FLOOR LEAK  denoised.min  (clean floor ~0; prior figure -0.0017)')
print('  mean {floor_leak_mean:+.5f} | most negative {floor_leak_min:+.5f}'.format(**s))
print('\nINVENTED STRUCTURE  (background pixels pushed above {:.0%} of clean peak,'
      ' blobs >= {} px)'.format(INVENT_FRAC, BLOB_MIN_PX))
print('  channels with >=1 fake blob: {:.0f}% | blobs/channel {:.2f}'.format(
    100 * s['frac_channels_with_blob'], s['blobs_per_channel']))
print('  mean invented area fraction of background: {invented_frac_mean:.4%} |'
      ' worst channel: {invented_frac_max:.4%}'.format(**s))

if 'snr_median' in s:
    print('\n  SNR split at median {snr_median:.1f}  ->  is invented structure low-SNR specific?'.format(**s))
    print('    low-SNR  half: blobs/channel {low_snr_blobs_per_channel:.2f} |'
          ' invented area {low_snr_invented_frac:.4%} | overshoot {low_snr_overshoot:.3f}'.format(**s))
    print('    high-SNR half: blobs/channel {high_snr_blobs_per_channel:.2f} |'
          ' invented area {high_snr_invented_frac:.4%} | overshoot {high_snr_overshoot:.3f}'.format(**s))

ov   = np.array([r['overshoot'] for r in rows])
leak = np.array([r['floor_leak'] for r in rows])
inv   = np.array([r['invented_frac'] for r in rows])
snr  = np.array([r['snr'] for r in rows])

diag_csv = os.path.join(OUT_DIR, 'artifact_diagnostics_sweepwinner.csv')
with open(diag_csv, 'w', newline='') as cf:
    w = csv.DictWriter(cf, fieldnames=['cube', 'channel', 'snr', 'overshoot', 'floor_leak',
                                       'invented_frac', 'invented_blobs'])
    w.writeheader()
    for r in rows: w.writerow(r)
print('\nper-channel diagnostics saved ->', diag_csv)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(ov, bins=25, color='#4E91C7', alpha=0.85)
ax[0].axvline(1.0, color='k', ls='--', label='no overshoot')
ax[0].axvline(1.151, color='#E8715A', ls=':', label='prior ch-100 figure')
ax[0].set_xlabel('denoised.max / clean.max'); ax[0].set_ylabel('channels')
ax[0].set_title('Peak overshoot'); ax[0].legend(fontsize=8)
ax[1].scatter(snr, inv * 100, s=18, alpha=0.7, color='#4E91C7')
ax[1].set_xlabel('channel SNR (clean peak / background std)')
ax[1].set_ylabel('invented background area (%)')
ax[1].set_title('Invented structure vs SNR'); ax[1].set_xscale('log')
ax[2].hist(leak, bins=25, color='#9B9B9B', alpha=0.9)
ax[2].axvline(0.0, color='k', ls='--')
ax[2].set_xlabel('denoised.min'); ax[2].set_title('Negative floor leak')
for a in ax: a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'artifact_diagnostics_sweepwinner.png'), dpi=140); plt.show()

## 8. Worst offenders — visual check of the invented structure

The three validation channels with the largest invented-area fraction, shown dirty / denoised /
clean. If the diagnostic in Section 7 is measuring something real, the fake structure should be
visible here; if these look fine, the thresholds need revisiting before the number is quoted.

In [ ]:
worst = sorted(rows, key=lambda r: -r['invented_frac'])[:3]
key = {(r['cube'], r['channel']) for r in worst}
picks = [ix for ix in range(len(val_ds))
         if (val_ds.cube_paths[val_ds.index[ix][0]][2], int(val_ds.index[ix][1])) in key][:3]

fig, ax = plt.subplots(len(picks), 3, figsize=(11, 3.6 * len(picks)), squeeze=False)
for k, t in enumerate(['dirty', 'denoised', 'clean GT']):
    ax[0][k].set_title(t, fontweight='bold')
with torch.no_grad():
    for r, ix in enumerate(picks):
        d, c = val_ds[ix]
        pred = eval_net(d[None].to(device),
                        torch.zeros(1, dtype=torch.long, device=device))[0, 0].cpu().numpy()
        cl = c[0].numpy(); vmax = float(cl.max())
        ci, ch = val_ds.index[ix]
        info = next(q for q in rows
                    if q['cube'] == val_ds.cube_paths[ci][2] and q['channel'] == int(ch))
        for col, im in enumerate([d[0].numpy(), pred, cl]):
            ax[r][col].imshow(np.clip(im, 0, vmax), cmap='inferno', vmin=0, vmax=vmax)
            ax[r][col].axis('off')
        ax[r][0].set_title('{} ch {}\nSNR {:.1f} | invented {:.3%} | {} blob(s)'.format(
            val_ds.cube_paths[ci][2], ch, info['snr'], info['invented_frac'],
            info['invented_blobs']), fontsize=8, loc='left')
fig.suptitle('Worst invented-structure channels (shared colour scale = clean peak)',
             fontweight='bold', y=1.0)
plt.tight_layout()
os.makedirs('../experiments', exist_ok=True)
plt.savefig('../experiments/hallucination_worst_channels.png', dpi=140); plt.show()
print('saved -> experiments/hallucination_worst_channels.png')

## 9. Persist artifacts to /kaggle/working (survives session end)

Copies the checkpoint and every CSV to the top level of `/kaggle/working` (outside the git
clone) so `kaggle kernels output <slug>` retrieves them. **The CSVs must be pulled down and
committed to `results/` manually** — Kaggle's GitHub sync pushes only the `.ipynb`, not files
written during the session.

In [ ]:
import shutil
if ON_KAGGLE:
    for src_p in [CKPT_WINNER,
                  '../results/moment_map_holdout_summary_sweepwinner.csv',
                  '../results/artifact_diagnostics_sweepwinner.csv']:
        if os.path.exists(src_p):
            dst = '/kaggle/working/' + os.path.basename(src_p)
            shutil.copy2(src_p, dst)
            print('persisted ->', dst, f'({os.path.getsize(dst)/1e6:.1f} MB)')
    print('\nremember: download the CSVs from the kernel Output tab and commit them to results/')
else:
    print('not on Kaggle -- nothing to persist')